# Myeloid cell_type_L3_refined Patch
**Date:** 2026-03-29  
**Purpose:** Three targeted updates to `cell_type_L3_refined`:
1. **NaN fill** — 2029 Non-classical monocytes → `Non-classical monocytes`
2. **Drop** — cells labeled `Low-quality Interstitial macrophages`
3. **Collapse** — all `* Interstitial macrophages` except `CD163L1+` and `Inflammatory` → `Interstitial macrophages`

In [1]:
# ===== 0. Imports & Config =====
import scanpy as sc
import pandas as pd
import numpy as np
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

INPUT_PATH  = Path("/home/h2048/data/py/0209/myeloid_validation_optimized/adata_myeloid_refined_FINAL.h5ad")
OUTPUT_PATH = Path("/home/h2048/data/py/0329/adata_myeloid_L3refined_patched_v1.h5ad")
OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)

COL = 'cell_type_L3_refined'
print(f"Input : {INPUT_PATH}")
print(f"Output: {OUTPUT_PATH}")

Input : /home/h2048/data/py/0209/myeloid_validation_optimized/adata_myeloid_refined_FINAL.h5ad
Output: /home/h2048/data/py/0329/adata_myeloid_L3refined_patched_v1.h5ad


In [2]:
# ===== 1. Load =====
adata = sc.read_h5ad(INPUT_PATH)
print(f"Loaded: {adata.shape}")
print(f"\nBefore patch — {COL} value counts:")
print(adata.obs[COL].value_counts(dropna=False).to_string())

Loaded: (54731, 35112)

Before patch — cell_type_L3_refined value counts:
cell_type_L3_refined
Resident Alveolar macrophages                    20433
Inflammatory Interstitial macrophages             5092
M2-like Interstitial macrophages                  4918
Neutrophils                                       4054
Atypically activated Interstitial macrophages     3882
Mast cells                                        3726
Resting Alveolar macrophages                      3711
Typical Classical monocytes                       3675
NaN                                               2029
Inflammatory Classical monocytes                  1179
Conventional cDC2                                  866
Langerhans-like cDC2                               445
pDC                                                335
CD163L1+ Interstitial macrophages                  232
Low-quality Interstitial macrophages               102
Immunoregulatory Interstitial macrophages           52


In [3]:
# ===== 2. Safety checks =====
# 2a. Confirm all NaN are Non-classical monocytes in L2
nan_mask = adata.obs[COL].isna()
l2_of_nan = adata.obs.loc[nan_mask, 'cell_type_L2'].unique()
assert list(l2_of_nan) == ['Non-classical monocytes'], \
    f"Unexpected L2 labels under NaN: {l2_of_nan}"
print(f"[OK] All {nan_mask.sum()} NaN cells are Non-classical monocytes in L2")

# 2b. Confirm Low-quality cells to be dropped
lq_mask = adata.obs[COL] == 'Low-quality Interstitial macrophages'
print(f"[OK] Low-quality Interstitial macrophages to drop: {lq_mask.sum()} cells")

# 2c. List all Interstitial macrophages subtypes
im_labels = [x for x in adata.obs[COL].dropna().unique() if 'Interstitial macrophages' in str(x)]
print(f"\nAll Interstitial macrophages subtypes found:")
for lbl in sorted(im_labels):
    n = (adata.obs[COL] == lbl).sum()
    print(f"  {lbl!r:55s}  n={n}")

[OK] All 2029 NaN cells are Non-classical monocytes in L2
[OK] Low-quality Interstitial macrophages to drop: 102 cells

All Interstitial macrophages subtypes found:
  'Atypically activated Interstitial macrophages'          n=3882
  'CD163L1+ Interstitial macrophages'                      n=232
  'Immunoregulatory Interstitial macrophages'              n=52
  'Inflammatory Interstitial macrophages'                  n=5092
  'Low-quality Interstitial macrophages'                   n=102
  'M2-like Interstitial macrophages'                       n=4918


In [4]:
# ===== 3. Apply patch =====
# Work on a string Series (category -> object for easy mutation)
col_patched = adata.obs[COL].astype(object).copy()

# --- 3a. Fill NaN ---
col_patched.loc[nan_mask] = 'Non-classical monocytes'
print(f"[3a] Filled {nan_mask.sum()} NaN -> 'Non-classical monocytes'")

# --- 3b. Collapse Interstitial macrophages subtypes ---
# Keep: 'CD163L1+ Interstitial macrophages'
# Keep: 'Inflammatory Interstitial macrophages'
# Collapse everything else that contains 'Interstitial macrophages'
# (Low-quality will be handled by drop in step 3c, but collapse first is fine;
#  they won't survive the drop anyway)
KEEP_IM = {
    'CD163L1+ Interstitial macrophages',
    'Inflammatory Interstitial macrophages',
}
collapse_mask = (
    col_patched.str.contains('Interstitial macrophages', na=False)
    & (~col_patched.isin(KEEP_IM))
)
n_collapsed = collapse_mask.sum()
col_patched.loc[collapse_mask] = 'Interstitial macrophages'
print(f"[3b] Collapsed {n_collapsed} cells -> 'Interstitial macrophages'")

# --- 3c. Drop Low-quality ---
# After collapse, 'Low-quality Interstitial macrophages' is now 'Interstitial macrophages'
# We need to track the original lq_mask (computed before collapse) to drop those cells
keep_mask = ~lq_mask
print(f"[3c] Dropping {lq_mask.sum()} Low-quality Interstitial macrophages cells")

# Write back before subsetting
adata.obs[COL] = col_patched.astype('category')

# Subset
adata = adata[keep_mask].copy()
print(f"\nShape after drop: {adata.shape}")

[3a] Filled 2029 NaN -> 'Non-classical monocytes'
[3b] Collapsed 8954 cells -> 'Interstitial macrophages'
[3c] Dropping 102 Low-quality Interstitial macrophages cells

Shape after drop: (54629, 35112)


In [5]:
# ===== 4. Verify =====
print(f"After patch — {COL} value counts:")
vc = adata.obs[COL].value_counts(dropna=False)
print(vc.to_string())

# Assertions
assert adata.obs[COL].isna().sum() == 0, "NaN still present!"
assert 'Low-quality Interstitial macrophages' not in adata.obs[COL].values, "Low-quality still present!"
assert 'Non-classical monocytes' in adata.obs[COL].values, "Non-classical monocytes missing!"
assert 'Interstitial macrophages' in adata.obs[COL].values, "Interstitial macrophages missing!"
assert 'CD163L1+ Interstitial macrophages' in adata.obs[COL].values, "CD163L1+ lost!"

# Confirm no other '* Interstitial macrophages' besides the three expected
im_remaining = sorted([x for x in adata.obs[COL].unique() if 'Interstitial macrophages' in str(x)])
expected_im  = sorted(['CD163L1+ Interstitial macrophages',
                        'Inflammatory Interstitial macrophages',
                        'Interstitial macrophages'])
assert im_remaining == expected_im, \
    f"Unexpected IM labels: {im_remaining}"

print("\n[ALL ASSERTIONS PASSED]")

After patch — cell_type_L3_refined value counts:
cell_type_L3_refined
Resident Alveolar macrophages            20433
Interstitial macrophages                  8852
Inflammatory Interstitial macrophages     5092
Neutrophils                               4054
Mast cells                                3726
Resting Alveolar macrophages              3711
Typical Classical monocytes               3675
Non-classical monocytes                   2029
Inflammatory Classical monocytes          1179
Conventional cDC2                          866
Langerhans-like cDC2                       445
pDC                                        335
CD163L1+ Interstitial macrophages          232

[ALL ASSERTIONS PASSED]


In [6]:
# ===== 5. Updated L2 x L3_refined summary =====
ct = pd.crosstab(adata.obs['cell_type_L2'], adata.obs[COL])
pct = ct.div(ct.sum(axis=1), axis=0) * 100
pct.insert(0, 'Total_cells', ct.sum(axis=1))
print("=== Updated L2 -> L3_refined composition (%) ===")
print(pct.round(1).to_string())

=== Updated L2 -> L3_refined composition (%) ===
cell_type_L3_refined     Total_cells  CD163L1+ Interstitial macrophages  Conventional cDC2  Inflammatory Classical monocytes  Inflammatory Interstitial macrophages  Interstitial macrophages  Langerhans-like cDC2  Mast cells  Neutrophils  Non-classical monocytes  Resident Alveolar macrophages  Resting Alveolar macrophages  Typical Classical monocytes    pDC
cell_type_L2                                                                                                                                                                                                                                                                                                                                                          
Alveolar macrophages           24144                                0.0                0.0                               0.0                                    0.0                       0.0                   0.0         0.0          

In [7]:
# ===== 6. Save =====
adata.write_h5ad(OUTPUT_PATH, compression='gzip', compression_opts=9)
print(f"Saved: {OUTPUT_PATH}")
print(f"Final shape: {adata.shape}")

Saved: /home/h2048/data/py/0329/adata_myeloid_L3refined_patched_v1.h5ad
Final shape: (54629, 35112)
